# Lab 04: File formats and data layout

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>2.1 Why is CSV heavy?</b></span><br><span style="color:#d93025"><b>2.2 Columnar storage</b></span><br><span style="color:#d93025"><b>2.3 Schema evolution</b></span><br><span style="color:#d93025"><b>2.4 Predicate pushdown</b></span><br><span style="color:#d93025"><b>2.5 Small file problem</b></span><br><span style="color:#d93025"><b>2.6 Avro for Kafka</b></span><br><span style="color:#d93025"><b>2.7 Disadvantages of Parquet</b></span><br><span style="color:#d93025"><b>2.8 Partitioned folders</b></span><br><span style="color:#d93025"><b>2.9 Splittable files</b></span><br><span style="color:#d93025"><b>2.10 Parquet vs ORC</b></span>

**Goal:** write the same data in five formats, prove pushdown and pruning in the plan, and see what partition folders, small files and gzip do. The last section is date and complex-type practice.

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question and run it. The answers are at the very end of this notebook (collapsed): open one only after you tried. Each answer shows the code and the result of the reference run, so you can compare your numbers.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then open the answer of that question.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Generic load/save functions](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html)
* [Parquet files](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html)
* [ORC files](https://spark.apache.org/docs/latest/sql-data-sources-orc.html)
* [Avro data source](https://spark.apache.org/docs/latest/sql-data-sources-avro.html)
* [Parquet configuration (filter pushdown)](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#configuration)
* [Parquet file format](https://parquet.apache.org/docs/file-format/)
* [pyarrow ParquetFile](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetFile.html)
* [Partition discovery](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#partition-discovery)
* [Performance tuning: partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
* [DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)
* [CSV options (compression)](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)
* [Schema merging](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging)
* [functions.date_trunc](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_trunc.html)
* [functions.sequence](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sequence.html)
* [functions.explode](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)
* [functions.collect_set](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_set.html)
* [functions.array_sort](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_sort.html)
* [Chapter 2: A Tour of PySpark Data Types](https://spark.apache.org/docs/latest/api/python/user_guide/touroftypes.html)
* [functions.from_json](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day04_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session(avro=True)
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

## A. Five formats, one dataset

**4.1 Write *orders* as CSV (with header), JSON, Parquet, ORC and Avro into `formats/<format>`. Which is the biggest, and which is the fastest to read?**
* *sizes* = dict `{format: folder size in MB}` (use `folder_mb(path)`)
* *read_times* = dict `{format: seconds}` for reading it back (CSV and JSON with `orders.schema`) and computing `sum(amount)`, timed with `timeit`

**4.2 What does Spark push down and read for `parquet_df.filter(amount > 900).select("order_id", "amount")`?** Load the Parquet copy as *parquet_df*. Copy the *PushedFilters* from the plan into *pushed* (list of strings) and the columns in *ReadSchema* into *read_cols* (sorted names).

**4.3 Why does sorted data skip more? Look inside the Parquet footer.** Write *orders* twice as a single file with small row groups (`option("parquet.block.size", 1024 * 1024)`, `coalesce(1)`): once as it is and once sorted by *amount*. With `pyarrow.parquet.ParquetFile(file).metadata`, count the row groups whose largest *amount* (column statistics, `max`) is at most 990, because a filter `amount > 990` could skip them.

Put the numbers in *unsorted_skip*, *sorted_skip* and the number of row groups in *n_groups*.

## B. Layout: partition folders, small files, splittable files

**4.4 How do partition folders help?** Write *orders* with `partitionBy("order_date")` into `layout/by_date`. How many `order_date=...` folders are there (*n_folders*)? Then read it back and filter one day (`2024-02-14`): copy the *PartitionFilters* of the scan into *partition_filter* (a string).

**4.5 Create the small file problem, then fix it.** Write *orders* partitioned by *order_date* twice:
* `layout/small`: after `repartition(50)`
* `layout/fixed`: after `repartition("order_date")`

Count the Parquet files of each (*files_small*, *files_fixed*, files are in `layout/<name>/order_date=.../`) and the read partitions of each (*parts_small*, *parts_fixed*, from `rdd.getNumPartitions()`).

**4.6 Which files can be split into several read tasks?** Write *orders* as one CSV file (`coalesce(1)`, with header), once plain (`csv_none`) and once gzip-compressed (`option("compression", "gzip")`, `csv_gzip`). How many read partitions does each have (*plain_parts*, *gzip_parts*)? Then set `spark.sql.files.maxPartitionBytes` to `16m` and count the plain one again (*plain_16*). Reset the setting afterwards.

**4.7 What does schema evolution look like in Parquet?** Write the 1,000 orders with `order_id <= 1000` to `evo/batch=1`, and the same orders plus a new column *discount* (value `0.1`) to `evo/batch=2`. Read the folder `evo` with `option("mergeSchema", "true")` into *merged*. Put the column names in *merged_cols* and the number of batch-1 rows where *discount* is null in *batch1_nulls*.

**4.8 Which filters can not be pushed down, and how do you fix them?** On *parquet_df*, copy the *PushedFilters* (sorted lists, `[]` if none) of these four filters into a dict *pushed_by*:
* `lower_status`: `F.lower("status") == "paid"`
* `cast_amount`: `F.col("amount").cast("string") == "5.99"`
* `raw_status`: `F.col("status") == "paid"`
* `raw_amount`: `F.col("amount") == 5.99`

## C. Coding practice: dates and complex types

Four problems with date functions, a date spine, arrays and JSON. Write them from scratch. (Tip: collect dates or text, not timestamps, so your answer does not depend on your time zone.)

**4.9 Weekly revenue.** Group the *paid* orders by week (`date_trunc("week", "order_ts")`, weeks start on Monday; cast the week to a date) and sum the *amount*. Put `[number of weeks, the week with the highest revenue as text YYYY-MM-DD, its revenue rounded to 2 decimals]` in *result*.

**4.10 Fill the missing days.** Count the *returned* orders paid by *wallet* with an *amount* over 990 for every day from `2024-01-01` to `2024-03-31` (91 days), including the days with no such order (count 0). Build the list of days with `sequence` and `explode`, then left join. Put `[days_without, max_in_a_day]` in *result*: the number of days with a count of 0 and the largest daily count.

**4.11 Which product categories did each customer return?** For the customers 2 to 6, collect the distinct *category* of their *returned* orders into a sorted array. *cats* = list of `(customer_id, [categories...])` sorted by *customer_id*.

**4.12 Parse JSON lines.** *raw* has one JSON text per row (one of them is not JSON at all). Parse the column *line* with `from_json` and the schema `order_id long, items array<struct<sku:string,qty:int>>`. How many rows could not be parsed (*n_bad*), and what is the total *qty* over all items of the good rows (*total_qty*)? Put `[total_qty, n_bad]` in *result*.

In [ ]:
raw = spark.createDataFrame([
    ('{"order_id": 1, "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]}',),
    ('{"order_id": 2, "items": [{"sku": "a", "qty": 5}]}',),
    ('{"order_id": 3, "items": []}',),
    ('not json at all',),
], "line string")   # given


---
Done? In the Theory part of Day 4, answer the red checklist topics aloud, with the sizes, plans and file counts from this lab as evidence.

## Answers
Try every question yourself first. When you are stuck, click the arrow of that question below to open its answer, then compare it with your own. Each answer is the reference solution with the result of its run; copy the code into the cell above to run it.

<details>
<summary><b>4.1</b> &nbsp;Write orders as CSV (with header), JSON, Parquet, ORC and Avro into formats/&lt;format&gt;.</summary>

```python
sizes, read_times = {}, {}
for fmt in ["csv", "json", "parquet", "orc", "avro"]:
    path = f"formats/{fmt}"
    shutil.rmtree(path, ignore_errors=True)
    writer = orders.write.option("header", True) if fmt == "csv" else orders.write
    writer.format(fmt).save(path)
    sizes[fmt] = folder_mb(path)
    reader = spark.read.format(fmt)
    if fmt in ("csv", "json"):
        reader = reader.option("header", True).schema(orders.schema)
    read_times[fmt] = timeit(lambda: reader.load(path).agg(F.sum("amount")).collect())
sizes, read_times
```

Result:

```
({'csv': 146.9, 'json': 346.8, 'parquet': 56.6, 'orc': 36.4, 'avro': 71.3},
 {'csv': 0.93, 'json': 1.04, 'parquet': 0.17, 'orc': 0.29, 'avro': 0.34})
```

</details>

<details>
<summary><b>4.2</b> &nbsp;What does Spark push down and read for parquet_df.filter(amount &gt; 900).select("order_id" ...</summary>

```python
parquet_df = spark.read.parquet("formats/parquet")
q = parquet_df.filter(F.col("amount") > 900).select("order_id", "amount")
text = plan(q)
pushed = re.search(r"PushedFilters: \[(.*?)\]", text).group(1).split(", ")
read_cols = sorted(c.split(":")[0] for c in re.search(r"ReadSchema: struct<(.*)>", text).group(1).split(","))
pushed, read_cols
```

Result:

```
(['IsNotNull(amount)', 'GreaterThan(amount,900.0)'], ['amount', 'order_id'])
```

</details>

<details>
<summary><b>4.3</b> &nbsp;Why does sorted data skip more? Look inside the Parquet footer.</summary>

```python
import glob
import pyarrow.parquet as pq

def skippable(df, name):
    path = f"footers/{name}"
    shutil.rmtree(path, ignore_errors=True)
    df.coalesce(1).write.option("parquet.block.size", 1024 * 1024).parquet(path)
    md = pq.ParquetFile(glob.glob(path + "/*.parquet")[0]).metadata
    col = md.schema.names.index("amount")
    return sum(1 for i in range(md.num_row_groups) if md.row_group(i).column(col).statistics.max <= 990), md.num_row_groups

unsorted_skip, n_groups = skippable(orders, "unsorted")
sorted_skip, _ = skippable(orders.orderBy("amount"), "sorted")
unsorted_skip, sorted_skip, n_groups
```

Result:

```
(0, 152, 154)
```

</details>

<details>
<summary><b>4.4</b> &nbsp;How do partition folders help?</summary>

```python
shutil.rmtree("layout/by_date", ignore_errors=True)
orders.write.partitionBy("order_date").parquet("layout/by_date")
n_folders = len([d for d in os.listdir("layout/by_date") if d.startswith("order_date=")])
one_day = spark.read.parquet("layout/by_date").filter(F.col("order_date") == "2024-02-14").select("order_id", "amount")
partition_filter = re.search(r"PartitionFilters: \[(.*?)\]", plan(one_day)).group(1)
n_folders, partition_filter
```

Result:

```
(91, 'isnotnull(order_date#244), (order_date#244 = 2024-02-14)')
```

</details>

<details>
<summary><b>4.5</b> &nbsp;Create the small file problem, then fix it.</summary>

```python
import glob
for name, df in [("small", orders.repartition(50)), ("fixed", orders.repartition("order_date"))]:
    shutil.rmtree(f"layout/{name}", ignore_errors=True)
    df.write.partitionBy("order_date").parquet(f"layout/{name}")
files_small = len(glob.glob("layout/small/*/*.parquet"))
files_fixed = len(glob.glob("layout/fixed/*/*.parquet"))
parts_small = spark.read.parquet("layout/small").rdd.getNumPartitions()
parts_fixed = spark.read.parquet("layout/fixed").rdd.getNumPartitions()
files_small, files_fixed, parts_small, parts_fixed
```

Result:

```
(4550, 91, 143, 4)
```

</details>

<details>
<summary><b>4.6</b> &nbsp;Which files can be split into several read tasks?</summary>

```python
for comp in ["none", "gzip"]:
    path = f"csv_{comp}"
    shutil.rmtree(path, ignore_errors=True)
    orders.coalesce(1).write.option("header", True).option("compression", comp).csv(path)
readp = lambda path: spark.read.option("header", True).csv(path).rdd.getNumPartitions()
plain_parts, gzip_parts = readp("csv_none"), readp("csv_gzip")
spark.conf.set("spark.sql.files.maxPartitionBytes", "16m")
plain_16 = readp("csv_none")
spark.conf.unset("spark.sql.files.maxPartitionBytes")
plain_parts, gzip_parts, plain_16
```

Result:

```
(4, 1, 9)
```

</details>

<details>
<summary><b>4.7</b> &nbsp;What does schema evolution look like in Parquet?</summary>

```python
shutil.rmtree("evo", ignore_errors=True)
few = orders.filter(F.col("order_id") <= 1000)
few.write.parquet("evo/batch=1")
few.withColumn("discount", F.lit(0.1)).write.parquet("evo/batch=2")
merged = spark.read.option("mergeSchema", "true").parquet("evo")
merged_cols = merged.columns
batch1_nulls = merged.filter("batch = 1 AND discount IS NULL").count()
merged_cols, batch1_nulls
```

Result:

```
(['order_id',
  'customer_id',
  'customer_name',
  'country',
  'segment',
  'product_id',
  'category',
  'price',
  'quantity',
  'amount',
  'status',
  'payment_method',
  'order_ts',
  'order_date',
...
```

</details>

<details>
<summary><b>4.8</b> &nbsp;Which filters can not be pushed down, and how do you fix them?</summary>

```python
def pushed_of(df):
    m = re.search(r"PushedFilters: \[(.*?)\]", plan(df))
    return sorted(m.group(1).split(", ")) if m and m.group(1) else []

pushed_by = {
    "lower_status": pushed_of(parquet_df.filter(F.lower("status") == "paid")),
    "cast_amount": pushed_of(parquet_df.filter(F.col("amount").cast("string") == "5.99")),
    "raw_status": pushed_of(parquet_df.filter(F.col("status") == "paid")),
    "raw_amount": pushed_of(parquet_df.filter(F.col("amount") == 5.99)),
}
pushed_by
```

Result:

```
{'lower_status': ['IsNotNull(status)'],
 'cast_amount': ['IsNotNull(amount)'],
 'raw_status': ['EqualTo(status,paid)', 'IsNotNull(status)'],
 'raw_amount': ['EqualTo(amount,5.99)', 'IsNotNull(amount)']}
```

</details>

<details>
<summary><b>4.9</b> &nbsp;Weekly revenue.</summary>

```python
weekly = (orders.filter(F.col("status") == "paid")
          .groupBy(F.date_trunc("week", "order_ts").cast("date").alias("week"))
          .agg(F.sum("amount").alias("revenue")))
best = weekly.orderBy(F.desc("revenue")).first()
result = [weekly.count(), str(best["week"]), round(best["revenue"], 2)]
result
```

Result:

```
[13, '2024-02-12', 15765178.8]
```

</details>

<details>
<summary><b>4.10</b> &nbsp;Fill the missing days.</summary>

```python
rare = (orders.filter((F.col("status") == "returned") & (F.col("amount") > 990) & (F.col("payment_method") == "wallet"))
        .groupBy("order_date").count())
spine = spark.sql("SELECT explode(sequence(to_date('2024-01-01'), to_date('2024-03-31'), interval 1 day)) AS order_date")
filled = spine.join(rare, "order_date", "left").na.fill({"count": 0})
result = [filled.filter(F.col("count") == 0).count(), filled.agg(F.max("count")).first()[0]]
result
```

Result:

```
[51, 2]
```

</details>

<details>
<summary><b>4.11</b> &nbsp;Which product categories did each customer return?</summary>

```python
returned = (orders.filter((F.col("status") == "returned") & F.col("customer_id").between(2, 6))
            .groupBy("customer_id").agg(F.array_sort(F.collect_set("category")).alias("cats"))
            .orderBy("customer_id"))
cats = [(r["customer_id"], list(r["cats"])) for r in returned.collect()]
cats
```

Result:

```
[(2, ['toys']),
 (3, ['office', 'sports', 'toys']),
 (4, ['garden', 'office', 'sports', 'toys']),
 (5, ['garden', 'kitchen', 'sports', 'toys']),
 (6, ['beauty', 'kitchen', 'music', 'office', 'sports'])]
```

</details>

<details>
<summary><b>4.12</b> &nbsp;Parse JSON lines.</summary>

```python
raw = spark.createDataFrame([
    ('{"order_id": 1, "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]}',),
    ('{"order_id": 2, "items": [{"sku": "a", "qty": 5}]}',),
    ('{"order_id": 3, "items": []}',),
    ('not json at all',),
], "line string")
schema = "order_id long, items array<struct<sku:string,qty:int>>"
parsed = raw.withColumn("j", F.from_json("line", schema))
n_bad = parsed.filter(F.col("j").isNull()).count()
total_qty = parsed.filter(F.col("j").isNotNull()).select(F.explode("j.items").alias("item")).agg(F.sum("item.qty")).first()[0]
result = [total_qty, n_bad]
result
```

Result:

```
[8, 0]
```

</details>